# Lab 08-03 — Communities + map/reduce summarization

**Track 08 · GraphRAG** — turning the entity graph from Lab 01 into the *index* GraphRAG actually queries.

Lab 01 produced an entity graph; this lab turns it into the index GraphRAG actually queries. Community detection partitions the graph with the **Leiden** algorithm (networkx **Louvain** fallback) so densely connected entities land in the same community — unsupervised structure discovery that works over a whole corpus without a query in sight. Then a **map** step summarizes every community's internal relations into a few prose sentences, and a **reduce** step folds those into one global corpus summary.

```text
rag-mini-wikipedia (first 20 passages)
  -> OllamaLLM extraction -> build_entity_graph (tools/graph)
  -> detect_communities (tools/graphrag — Leiden / Louvain, SEED=42)
  -> community_summaries (tools/graphrag — map: one LLM call per community)
  -> global_summary (tools/graphrag — reduce: one global corpus summary)
  -> verification gate (--verify)
```


## Setup

This notebook mirrors `src/curriculum/08-graphrag/03-communities.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

One prerequisite: this lab talks to a **local LLM**. The repo's `OllamaLLM` adapter (`src/llms/ollama.py`) expects Ollama serving `qwen2.5-coder:7b` at `localhost:11434` — fully local, zero API quota:

```bash
ollama pull qwen2.5-coder:7b   # if you haven't already
```

From the terminal, the lab runs as:

```bash
python src/curriculum/08-graphrag/03-communities.py          # run + demo
python src/curriculum/08-graphrag/03-communities.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   langchain-ollama -> OllamaLLM chat model (llms/ollama.py)
#   pandas           -> passage loading from the parquet corpus
#   networkx         -> graph object + Louvain fallback (tools/graphrag)
%pip install langchain-ollama pandas networkx


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd  # noqa: E402

from llms.ollama import OllamaLLM  # noqa: E402
from tools.graph import build_entity_graph  # noqa: E402
from tools.graphrag import (  # noqa: E402
    community_summaries,
    detect_communities,
    global_summary,
)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 20` caps the corpus at a deterministic head — each passage costs one entity-extraction LLM call, so the pool size *is* the extraction budget. `MAX_COMMUNITIES = 6` is the map/reduce cap: the largest communities get summarized, bounding the map step at six LLM calls no matter how many communities Leiden finds. `SEED = 42` makes the community detection (Leiden / Louvain) reproducible across runs.


In [ ]:
# 1. Configuration
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 20  # deterministic head; each passage costs one extraction call
MAX_COMMUNITIES = 6  # map/reduce cap: largest communities get summarized
SEED = 42  # deterministic community detection (Leiden / Louvain)


## 2. Load — first N passages of rag-mini-wikipedia

The corpus is `rag-mini-wikipedia` — the same parquet Lab 01 used — and we take the first `N_PASSAGES` passages by file order. The head is deterministic, which matters twice: the extracted graph, and therefore the communities and every downstream summary, is identical across runs.


In [ ]:
# 2. Load — first N passages of the rag-mini-wikipedia corpus
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — build graph, detect communities, map/reduce summaries

The index recipe, in three timed stages:

- **Build** — `build_entity_graph` (`src/tools/graph`) runs one LLM extraction call per passage: entities, relations, and claims in, a networkx graph out. This is the only stage that touches every passage.
- **Detect** — `detect_communities` partitions the graph with sknetwork's Leiden (networkx Louvain fallback). Fully local, no model call — pure structure discovery, so the index is built without a query in sight.
- **Map/reduce** — `community_summaries` (map) compresses each community's internal relations into a few prose sentences, one LLM call per community; `global_summary` (reduce) folds them into a single global corpus summary. `OllamaLLM` serves both stages, fully local with zero API quota.

The whole index costs roughly one LLM call per passage plus one per summarized community.


In [ ]:
# 3. Experiment — build graph, detect communities, map/reduce summaries
def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    llm = OllamaLLM()  # local qwen2.5-coder:7b; map/reduce + extraction

    t0 = time.perf_counter()
    graph = build_entity_graph(passages, llm)
    build_s = time.perf_counter() - t0

    communities = detect_communities(graph, seed=SEED)
    t0 = time.perf_counter()
    summaries = community_summaries(
        llm, graph, communities, max_communities=MAX_COMMUNITIES
    )
    map_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    global_text = global_summary(llm, [s["summary"] for s in summaries])
    reduce_s = time.perf_counter() - t0

    covered = {node for community in communities for node in community}
    return {
        "graph": graph,
        "communities": communities,
        "summaries": summaries,
        "global_summary": global_text,
        "coverage": len(covered),
        "build_s": build_s,
        "map_s": map_s,
        "reduce_s": reduce_s,
    }


## 4. Demo

The demo prints the artifact the index will later be queried against: entity/community counts and coverage, the community size distribution, the map summaries, the reduced global summary, and per-stage timing. Coverage is the property that makes the index sound — every entity belongs to exactly one community, so no part of the corpus is invisible to a later local/global query.


In [ ]:
# 4. Demo — print the artifact
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-03 — Communities + map/reduce summarization")
    graph = exp["graph"]
    print(f"{graph.number_of_nodes()} entities, {len(exp['communities'])} "
          f"communities, coverage {exp['coverage']}/{graph.number_of_nodes()}")
    print("=" * 66)

    sizes = sorted((len(c) for c in exp["communities"]), reverse=True)
    print(f"\n[1] Community sizes (largest first): {sizes}")

    print(f"\n[2] Map — community summaries (top {MAX_COMMUNITIES} by size):")
    for i, entry in enumerate(exp["summaries"], start=1):
        print(f"    C{i} (size {entry['size']}): {entry['summary']}")

    print(f"\n[3] Reduce — global corpus summary:")
    print(f"    {exp['global_summary']}")

    print(f"\n[4] Timing: build graph {exp['build_s']:.1f}s, "
          f"map {exp['map_s']:.1f}s, reduce {exp['reduce_s']:.1f}s")

    print(f"\n[5] Takeaway")
    print("    Communities partition the corpus by structure, not by query.")
    print("    The map step compresses every community into a few sentences;")
    print("    the reduce step folds them into one global summary. Lab 04")
    print("    queries this index two ways: local (walk the graph from the")
    print("    question's entities) and global (rank the summaries).")


## 5. Verification gate

The lab ships a `--verify` gate: hard checks the index must clear — every entity covered by a community, communities disjoint, at least two communities, the summary cap respected, and no empty summaries. This is the same gate the `.py --verify` run enforces: it turns "the lab ran" into "the lab ran *correctly*" — the index is structurally sound before Lab 04 ever queries it.


In [ ]:
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    graph = exp["graph"]
    communities = exp["communities"]
    covered = exp["coverage"]

    checks.append(("communities cover every entity (no orphan nodes)",
                   covered == graph.number_of_nodes()))
    checks.append(("communities are disjoint",
                   sum(len(c) for c in communities) == covered))
    checks.append((f"at least 2 communities (got {len(communities)})",
                   len(communities) >= 2))
    checks.append((f"summaries cover {len(exp['summaries'])} communities "
                   f"(cap {MAX_COMMUNITIES})",
                   len(exp["summaries"]) <= MAX_COMMUNITIES))
    checks.append(("every summary is non-empty",
                   all(entry["summary"].strip()
                       for entry in exp["summaries"])))
    checks.append(("global summary is non-empty",
                   bool(exp["global_summary"].strip())))
    checks.append(("community sizes are consistent with members",
                   all(len(entry["members"]) == entry["size"]
                       for entry in exp["summaries"])))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

This is the slow cell: ~20 LLM extraction calls to build the graph, up to 6 community summaries, and 1 reduce call — a few minutes against a local `qwen2.5-coder:7b`. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
